In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load
import numpy as np # linear algebra
import os
# STEFANOS: Conditionally import Modin Pandas
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

#Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/home/dias-benchmarks/notebooks/mpwolke/just-you-wait-rishi-sunak/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/home/dias-benchmarks/notebooks/mpwolke/just-you-wait-rishi-sunak/input/latest-elected-uk-prime-minister-rishi-sunak/uk_pm.pkl
/home/dias-benchmarks/notebooks/mpwolke/just-you-wait-rishi-sunak/input/latest-elected-uk-prime-minister-rishi-sunak/uk_pm.feather
/home/dias-benchmarks/notebooks/mpwolke/just-you-wait-rishi-sunak/input/latest-elected-uk-prime-minister-rishi-sunak/uk_pm.parquet
/home/dias-benchmarks/notebooks/mpwolke/just-you-wait-rishi-sunak/input/latest-elected-uk-prime-minister-rishi-sunak/uk_pm.csv


In [2]:
import re
import string

import nltk
from nltk.probability import FreqDist
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
from nltk import pos_tag
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# STEFANOS: Unneeded
# from wordcloud import WordCloud
from tqdm.auto import tqdm
import matplotlib.style as style
import time
style.use('fivethirtyeight')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt_tab to /home/jieq/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/jieq/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jieq/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [3]:
### cell 0 ###

train = pd.read_parquet('/home/dias-benchmarks/notebooks/mpwolke/just-you-wait-rishi-sunak/input/latest-elected-uk-prime-minister-rishi-sunak/uk_pm.parquet')
factor = 1
train = train.sample(frac=factor, random_state=0)

In [4]:
### cell 1 ###

train.head()

,text,username,hashtags,created_at,user followers count,replycount,retweetcount,likecount,quotecount,language,quotedtweet,inReplyToTweetId,inReplyToUser,mentionedUsers
87491,@nadhimzahawi @RishiSunak @Conservatives You'v...,MJFGD,None,2022-10-24 15:58:01,2450,0,0,0,0,en,None,1.584536e+18,https://twitter.com/nadhimzahawi,@nadhimzahawi @RishiSunak @Conservatives You'v...
77655,@nadhimzahawi @RishiSunak @Conservatives Ladie...,marknew33597947,None,2022-10-24 16:36:40,21,0,0,13,0,en,None,1.584536e+18,https://twitter.com/nadhimzahawi,@nadhimzahawi @RishiSunak @Conservatives Ladie...
71352,@NicolaSturgeon @danielgoyal @RishiSunak Oh wo...,thaitattoo,None,2022-10-24 17:03:27,60,0,0,0,0,en,None,1.584539e+18,https://twitter.com/NicolaSturgeon,@NicolaSturgeon @danielgoyal @RishiSunak Oh wo...
18421,Indian mentality:\n\nWe are proud of Rishi Sun...,tillusci,"['Rishi', 'RishiSunakPM']",2022-10-24 15:13:06,626,0,1,5,0,en,None,NaN,None,Indian mentality:\n\nWe are proud of Rishi Sun...
13757,"@maggie_erewash @RishiSunak With respect, if #...",JulesT737,"['RishiSunak', 'LizTruss']",2022-10-20 20:02:12,632,1,2,7,0,en,None,1.583161e+18,https://twitter.com/maggie_erewash,"@maggie_erewash @RishiSunak With respect, if #..."


In [5]:
### cell 2 ###

df = pd.read_csv("/home/dias-benchmarks/notebooks/mpwolke/just-you-wait-rishi-sunak/input/latest-elected-uk-prime-minister-rishi-sunak/uk_pm.csv", delimiter=',', encoding='utf8')
pd.set_option('display.max_columns', None)
df.tail()

,id,text,username,hashtags,created_at,user followers count,replycount,retweetcount,likecount,quotecount,language,quotedtweet,inReplyToTweetId,inReplyToUser,mentionedUsers
113573,1584553348736421888,@JackBarbour_ @piersmorgan @RishiSunak The man...,aqeelone22,NaN,2022-10-24 14:32:12+00:00,1162,0,0,0,0,en,NaN,1.584532e+18,https://twitter.com/JackBarbour_,@JackBarbour_ @piersmorgan @RishiSunak The man...
113574,1584553348677525505,@Jacob_Rees_Mogg @RishiSunak Let's hope you're...,EddieThornley,NaN,2022-10-24 14:32:12+00:00,97,0,0,0,0,en,NaN,1.584539e+18,https://twitter.com/Jacob_Rees_Mogg,@Jacob_Rees_Mogg @RishiSunak Let's hope you're...
113575,1584553348597895168,@sajidjavid @RishiSunak Desperate call for a d...,pedleysjohn,['ToriesUnfitToGovern'],2022-10-24 14:32:12+00:00,2484,0,0,0,0,en,NaN,1.584401e+18,https://twitter.com/sajidjavid,@sajidjavid @RishiSunak Desperate call for a d...
113576,1584553348132397056,@SkyNews Rulling Party needs a strong oppositi...,KarobiVC,NaN,2022-10-24 14:32:12+00:00,392,0,0,0,0,en,NaN,1.584512e+18,https://twitter.com/SkyNews,@SkyNews Rulling Party needs a strong oppositi...
113577,1584553347998105600,@trussliz @RishiSunak Liz Truss has won the ma...,sands_hill,NaN,2022-10-24 14:32:12+00:00,4,0,0,0,0,en,NaN,1.584546e+18,https://twitter.com/trussliz,@trussliz @RishiSunak Liz Truss has won the ma...


In [6]:
### cell 3 ###

#Code by Leon Wolber https://www.kaggle.com/leonwolber/reddit-nlp-topic-modeling-prediction

print(len(train[train['likecount'] < 500]), 'tweets with less than 500 dislikes')
print(len(train[train['likecount'] > 500]), 'tweets with more than 500 dislikes')

113332 tweets with less than 500 dislikes
246 tweets with more than 500 dislikes


In [7]:
### cell 4 ###

#Code by Leon Wolber https://www.kaggle.com/leonwolber/reddit-nlp-topic-modeling-prediction

# video with the most comments

train_tmp = train[train['likecount'] == train['likecount'].max()]['text']
if not train_tmp.empty:
    print(train_tmp.iloc[0])

"Warmest congratulations @RishiSunak! As you become UK PM, I look forward to working closely together on global issues, and implementing Roadmap 2030. Special Diwali wishes to the 'living bridge' of UK Indians, as we transform our historic ties into a modern partnership."

In [8]:
### cell 5 ###

train["text"] = train["text"].astype(str).str.replace(":"," ", regex=False)

In [ ]:
### cell 6 ###

#Code by Leon Wolber https://www.kaggle.com/leonwolber/reddit-nlp-topic-modeling-prediction

def remove_line_breaks(text):
    text = text.replace('\r', ' ').replace('\n', ' ')
    return text

#remove punctuation
def remove_punctuation(text):
    re_replacements = re.compile("__[A-Z]+__")  # such as __NAME__, __LINK__
    re_punctuation = re.compile("[%s]" % re.escape(string.punctuation))
    '''Escape all the characters in pattern except ASCII letters and numbers'''
    tokens = word_tokenize(text)
    tokens_zero_punctuation = []
    for token in tokens:
        if not re_replacements.match(token):
            token = re_punctuation.sub(" ", token)
        tokens_zero_punctuation.append(token)
    return ' '.join(tokens_zero_punctuation)

def remove_special_characters(text):
    text = re.sub('[^a-zA-z0-9\s]', '', text)
    return text

def lowercase(text):
    text_low = [token.lower() for token in word_tokenize(text)]
    return ' '.join(text_low)

def remove_stopwords(text):
    stop = set(stopwords.words('english'))
    word_tokens = nltk.word_tokenize(text)
    text = " ".join([word for word in word_tokens if word not in stop])
    return text

#remobe one character words
def remove_one_character_words(text):
    '''Remove words from dataset that contain only 1 character'''
    text_high_use = [token for token in word_tokenize(text) if len(token)>1]      
    return ' '.join(text_high_use)   
    
#%%
# Stemming with 'Snowball stemmer" package
def stem(text):
    stemmer = nltk.stem.snowball.SnowballStemmer('english')
    text_stemmed = [stemmer.stem(token) for token in word_tokenize(text)]        
    return ' '.join(text_stemmed)

def lemma(text):
    wordnet_lemmatizer = WordNetLemmatizer()
    word_tokens = nltk.word_tokenize(text)
    text_lemma = " ".join([wordnet_lemmatizer.lemmatize(word) for word in word_tokens])       
    return ' '.join(text_lemma)


#break sentences to individual word list
def sentence_word(text):
    word_tokens = nltk.word_tokenize(text)
    return word_tokens
#break paragraphs to sentence token 
def paragraph_sentence(text):
    sent_token = nltk.sent_tokenize(text)
    return sent_token    


def tokenize(text):
    """Return a list of words in a text."""
    return re.findall(r'\w+', text)

def remove_numbers(text):
    no_nums = re.sub(r'\d+', '', text)
    return ''.join(no_nums)



def clean_text(text):
    _steps = [
    remove_line_breaks,
    remove_one_character_words,
    remove_special_characters,
    lowercase,
    remove_punctuation,
    remove_stopwords,
    stem,
    remove_numbers
]
    for step in _steps:
        text=step(text)
    return text   
#%%

train['clean_text'] = pd.Series([clean_text(i) for i in tqdm(train['text'])])

In [10]:
### cell 7 ###

words = train["clean_text"].values

In [11]:
### cell 8 ###

# Optimized and semantically equivalent to the original loop
ls = [str(i_1 := w) for w in words]

In [12]:
### cell 9 ###

ls[:5]

['nadhimzahawi rishisunak conserv fuck parti decad',
 'nadhimzahawi rishisunak conserv ladi gentleman welcom stage nadhim flip flop zahawi back least mps prime minist week within  hour one ask resign within  hour 😂😂😂 strong stabl compet',
 'nicolasturgeon danielgoy rishisunak oh wow know proper screw',
 'indian mental proud rishi sunak howev doubt sonia gandhi meanwhil noth origin countri rishi rishisunakpm',
 'maggi erewash rishisunak respect rishisunak want would last select process liztruss']

In [13]:
### cell 10 ###

#Code by Leon Wolber https://www.kaggle.com/leonwolber/reddit-nlp-topic-modeling-prediction

most_pop = train.sort_values('likecount', ascending =False)[['text', 'likecount']].head(12)

most_pop['target1'] = most_pop['likecount']/1000

In [14]:
### cell 11 ###

import collections.abc
#gensim aliases to be done manually.
collections.Iterable = collections.abc.Iterable
collections.Mapping = collections.abc.Mapping
collections.MutableSet = collections.abc.MutableSet
collections.MutableMapping = collections.abc.MutableMapping
import gensim
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS
from nltk.stem import WordNetLemmatizer, SnowballStemmer
from nltk.stem.porter import *
import numpy as np
np.random.seed(2018)
import nltk

In [15]:
### cell 12 ###

stemmer = SnowballStemmer('english')

In [16]:
### cell 13 ###

train['text'].iloc[2]

'@NicolaSturgeon @danielgoyal @RishiSunak Oh wow, now we know we are properly screwed.'

In [17]:
### cell 14 ###

#Code by Leon Wolber https://www.kaggle.com/leonwolber/reddit-nlp-topic-modeling-prediction

#Code by Leon Wolber https://www.kaggle.com/leonwolber/reddit-nlp-topic-modeling-prediction

def lemmatize_stemming(text):
    return stemmer.stem(WordNetLemmatizer().lemmatize(text, pos='v'))

def preprocess(text):
    result = []
    for token in gensim.utils.simple_preprocess(text):
        if token not in gensim.parsing.preprocessing.STOPWORDS and len(token) > 3:
            result.append(lemmatize_stemming(token))
    return result
    
doc_sample = train['text'].iloc[1]
print('original document: ')

words = []

for word in doc_sample.split(' '):
    words.append(word)
    
    
print(words)
print('\n\n tokenized and lemmatized document: ')
print(preprocess(doc_sample))

original document: 
['@nadhimzahawi', '@RishiSunak', '@Conservatives', 'Ladies', 'and', 'Gentleman,welcome', 'to', 'the', 'stage,Nadhim', '"flip', 'flop"Zahawi,he', 'has', 'backed', 'at', 'least', '3', 'MPs', 'to', 'be', 'prime', 'minister', 'this', 'week', '2', 'of', 'them', 'within', '24', 'hours.[one', 'of', 'them', 'he', 'asked', 'to', 'resign', 'within', '24', 'hours]😂😂😂,strong,stable', 'and', 'competent!']


 tokenized and lemmatized document: 
['nadhimzahawi', 'rishisunak', 'conserv', 'ladi', 'gentleman', 'welcom', 'stage', 'nadhim', 'flip', 'flop', 'zahawi', 'back', 'prime', 'minist', 'week', 'hour', 'ask', 'resign', 'hour', 'strong', 'stabl', 'compet']


In [18]:
### cell 15 ###

train['text'] = train['text'].astype(str)

In [19]:
### cell 16 ###

#Code by Leon Wolber https://www.kaggle.com/leonwolber/reddit-nlp-topic-modeling-prediction

words = []

for i_2 in train['text']:
        words.append(i_2.split(' '))

In [20]:
### cell 17 ###

#Code by Leon Wolber https://www.kaggle.com/leonwolber/reddit-nlp-topic-modeling-prediction

dictionary = gensim.corpora.Dictionary(words)

count = 0
for k, v in dictionary.iteritems():
    print(k, v)
    count += 1
    if count > 10:
        break

0 @Conservatives
1 @RishiSunak
2 @nadhimzahawi
3 You've
4 decades
5 for
6 fucked
7 party
8 your
9 "flip
10 2


In [21]:
### cell 18 ###

bow_corpus = [dictionary.doc2bow(doc) for doc in words]
bow_corpus[4310]

[(1, 1),
 (104, 1),
 (162, 1),
 (392, 1),
 (2951, 1),
 (5240, 1),
 (10690, 1),
 (11004, 1),
 (19737, 1)]

In [22]:
### cell 19 ###

#Code by Leon Wolber https://www.kaggle.com/leonwolber/reddit-nlp-topic-modeling-prediction

bow_doc_4310 = bow_corpus[4310]

for i_3 in range(len(bow_doc_4310)):
    print("Word {} (\"{}\") appears {} time.".format(bow_doc_4310[i_3][0], 
                                               dictionary[bow_doc_4310[i_3][0]], 
bow_doc_4310[i_3][1]))

Word 1 ("@RishiSunak") appears 1 time.
Word 104 ("Sunak") appears 1 time.
Word 162 ("@MattHancock") appears 1 time.
Word 392 ("people") appears 1 time.
Word 2951 ("while") appears 1 time.
Word 5240 ("partied") appears 1 time.
Word 10690 ("died") appears 1 time.
Word 11004 ("alone.") appears 1 time.
Word 19737 ("out.
He") appears 1 time.
